
# `Nearshoring_tres_fuentes.ipynb` — Record linkage exportadores vs inversionistas

**Objetivo.** Cruzar la **Base 1** de exportadores con la **Base 2** de inversionistas (`fDiMarkets` + `Orbis Crossboarder`) para identificar qué exportadores aparecen como inversionistas y construir una correlativa auditable para traer datos de contacto.

**Variable principal de empalme.** Solo nombre de empresa:

- Base 1: `Empresa`
- Base 2 / fDiMarkets: `Company name`
- Base 2 / Orbis Crossboarder: `Company name Latin alphabet`

**Criterio técnico.** El país no se usa como llave de empalme. Se usa como control de calidad y bandera de revisión, porque en fDi el país puede representar sede/HQ del inversionista, no necesariamente el país de la filial exportadora. Esta distinción es crítica: si se exige país idéntico como bloqueo duro, se pierden matches reales de grupo/filial; si se ignora por completo, suben falsos positivos.

**Uso de `rues-linker`.** El notebook activa la librería ubicada en `ruta_paquete` y usa su `SimilarityCalculator` como una de las señales de similitud. Sobre esa señal se aplica una política conservadora y auditable de decisión para separar:

- `MATCH_SEGURO_MISMA_EMPRESA`
- `MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR`
- `MATCH_PROBABLE_REVISION_MANUAL`

**Advertencia.** Sin ground truth etiquetado no se puede garantizar precisión/recall real. Este notebook prioriza precisión y trazabilidad sobre recall agresivo. Los casos de país distinto o un solo token común no se promocionan automáticamente a match seguro.


In [1]:
!pip install -q xlsxwriter rapidfuzz openpyxl pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 7.1 MB/s eta 0:00:00



## A. Librerías, configuración y funciones base

Ejecute esta sección una vez por sesión. La configuración editable está al final, en la sección B.


In [2]:
from __future__ import annotations

import contextlib
import gc
import importlib
import io
import json
import logging
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from rapidfuzz import fuzz

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import drive  # type: ignore
    EN_COLAB = True
except Exception:
    drive = None
    EN_COLAB = False


def _default_ruta_paquete() -> Path:
    if EN_COLAB:
        return Path('/content/drive/MyDrive/ProColombia/record_linkage')
    return Path('/mnt/data/rues_unpacked/rues-linker-main')


def _default_dir_datos() -> Path:
    if EN_COLAB:
        return Path('/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29')
    return Path('/mnt/data')


@dataclass(frozen=True)
class ConfigNearshoring:
    fecha_procesamiento: str = '2026-06-29'
    etiqueta_corrida: str = 'NEARSHORING_TRES_FUENTES'
    seed: int = 42

    ruta_paquete: Path = field(default_factory=_default_ruta_paquete)
    dir_datos: Path = field(default_factory=_default_dir_datos)
    archivo_base1: str = 'Base 1_Empresas exportadoras a USA en 2025 por países y productos priorizados.xlsx'
    archivo_base2: str = 'Base 2_Descargas bases de inversión.xlsx'
    hoja_fdi: str = 'fDiMarkets'
    hoja_orbis: str = 'Orbis Crossboarder'
    header_fdi: int = 2
    hojas_excluir_base1: tuple[str, ...] = ('Criterios CRUCE',)

    # Columnas de empalme de negocio.
    col_base1_empresa: str = 'Empresa'
    col_fdi_empresa: str = 'Company name'
    col_orbis_empresa: str = 'Company name Latin alphabet'

    # Contrato técnico interno.
    col_nombre_match: str = 'RAZON_SOCIAL_MATCH'
    col_nombre_raw: str = 'RAZON_SOCIAL_RAW'
    col_nit_tecnico: str = 'NIT'

    # Configuración de normalización y scoring.
    usar_rues_similarity: bool = True
    min_largo_compacta_match: int = 4
    umbral_ratio_seguro: float = 94.0
    umbral_ratio_revision: float = 88.0
    umbral_rues_seguro: float = 0.96
    umbral_rues_revision: float = 0.90
    permitir_revision_un_token: bool = True
    permitir_match_seguro_un_token_exacto: bool = True

    # País no es llave de match, pero sí guardrail/auditoría.
    exigir_pais_compatible_para_seguro_juridico: bool = True

    # Salidas.
    exportar_excel: bool = True
    exportar_csv: bool = True
    exportar_parquet: bool = False
    max_muestras_revision: int = 200

    def __post_init__(self) -> None:
        object.__setattr__(self, 'ruta_paquete', Path(self.ruta_paquete))
        object.__setattr__(self, 'dir_datos', Path(self.dir_datos))


@dataclass(frozen=True)
class RutasNearshoring:
    config: ConfigNearshoring
    archivo_base1: Path
    archivo_base2: Path
    output_dir: Path
    excel_resultados: Path
    log_file: Path


def construir_rutas(config: ConfigNearshoring) -> RutasNearshoring:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out = config.dir_datos / f'salidas_{config.etiqueta_corrida.lower()}_{ts}'
    return RutasNearshoring(
        config=config,
        archivo_base1=config.dir_datos / config.archivo_base1,
        archivo_base2=config.dir_datos / config.archivo_base2,
        output_dir=out,
        excel_resultados=out / 'nearshoring_tres_fuentes_resultados.xlsx',
        log_file=out / 'logs' / 'nearshoring_tres_fuentes.log',
    )


def setup_logger(log_file: Path | None = None, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger('nearshoring_tres_fuentes')
    logger.setLevel(level)
    logger.handlers.clear()
    fmt = logging.Formatter('%(asctime)s [%(levelname)-7s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    if log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file, encoding='utf-8')
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    logger.propagate = False
    return logger


def montar_drive_con_retry(intentos: int = 3, espera_s: int = 5) -> bool:
    if not EN_COLAB:
        print('ℹ️ Entorno local detectado; no se monta Google Drive.')
        return False
    if os.path.ismount('/content/drive'):
        print('ℹ️ Drive ya estaba montado.')
        return True
    for intento in range(1, intentos + 1):
        try:
            drive.mount('/content/drive')
            print(f'✅ Drive montado (intento {intento}/{intentos}).')
            return True
        except Exception as exc:
            print(f'⚠️ Intento {intento}/{intentos} falló: {type(exc).__name__}: {exc}')
            if intento < intentos:
                time.sleep(espera_s)
            else:
                raise
    return False


def resolver_repo_rues_linker(ruta_paquete: Path) -> dict[str, Any]:
    ruta_paquete = Path(ruta_paquete)
    candidatos = [ruta_paquete, ruta_paquete / 'rues-linker-main', ruta_paquete / 'rues-linker']
    for cand in candidatos:
        if (cand / 'pyproject.toml').exists():
            return {'kind': 'repo', 'path': cand}
    for cand in candidatos:
        if (cand / 'record_linkage' / '__init__.py').exists():
            return {'kind': 'src_parent', 'path': cand}
        if (cand / 'src' / 'record_linkage' / '__init__.py').exists():
            return {'kind': 'repo', 'path': cand}
    if (ruta_paquete / '__init__.py').exists() and ruta_paquete.name == 'record_linkage':
        return {'kind': 'package_dir', 'path': ruta_paquete.parent}

    zips: list[Path] = []
    if ruta_paquete.is_file() and ruta_paquete.suffix.lower() == '.zip':
        zips.append(ruta_paquete)
    elif ruta_paquete.exists():
        zips.extend(sorted(ruta_paquete.glob('rues-linker*.zip')))
        zips.extend(sorted(ruta_paquete.glob('*rues*linker*.zip')))

    if zips:
        zip_path = zips[0]
        extract_dir = Path('/content/rues_linker_src') if EN_COLAB else Path('/tmp/rues_linker_src')
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        for cand in [extract_dir, *extract_dir.iterdir()]:
            if (cand / 'pyproject.toml').exists():
                return {'kind': 'repo', 'path': cand}
            if (cand / 'record_linkage' / '__init__.py').exists():
                return {'kind': 'src_parent', 'path': cand}
            if (cand / 'src' / 'record_linkage' / '__init__.py').exists():
                return {'kind': 'repo', 'path': cand}

    raise FileNotFoundError(
        'No pude resolver rues-linker. Ruta recibida: '
        f'{ruta_paquete}. Debe contener pyproject.toml, record_linkage/ o un zip rues-linker*.zip.'
    )


def instalar_rues_linker(ruta_paquete: Path) -> Path:
    info = resolver_repo_rues_linker(ruta_paquete)
    kind = info['kind']
    root = Path(info['path'])
    if kind == 'repo':
        cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{root}[excel]']
        print('▶ Instalando repo editable:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
        for cand in (root / 'src', root):
            if (cand / 'record_linkage' / '__init__.py').exists() and str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
    else:
        print(f'▶ Activando paquete local legacy: {root}')
        deps = [
            'pyarrow>=14.0', 'scikit-learn>=1.3', 'scipy>=1.10', 'datasketch>=1.6',
            'rapidfuzz>=3.0', 'unidecode>=1.3', 'tqdm>=4.66', 'pydantic>=2.0',
            'pyyaml>=6.0', 'openpyxl>=3.1', 'xlsxwriter>=3.1',
        ]
        if EN_COLAB:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
    importlib.invalidate_caches()
    import record_linkage
    print('✅ record_linkage importado desde:', Path(record_linkage.__file__).resolve())
    return root


PAIS_ES_A_EN = {
    'India': 'India',
    'Japón': 'Japan',
    'Japon': 'Japan',
    'Países Bajos': 'Netherlands',
    'Paises Bajos': 'Netherlands',
    'España': 'Spain',
    'Espana': 'Spain',
    'Italia': 'Italy',
    'Bélgica': 'Belgium',
    'Belgica': 'Belgium',
}

SUBSECTOR_POR_PARTIDA_PREFIX = {
    '3304': 'Cosméticos',
    '380891': 'Insecticidas',
    '4819': 'Envases y empaques de papel y cartón',
    '4823': 'Envases y empaques de papel y cartón',
    '854620': 'Aparatos eléctricos',
}

LEGAL_TOKENS = set(
    'SA SAS SAU SL SLU SRL SPA BV NV LTD LIMITED PRIVATE PVT CO COMPANY CORPORATION CORP INC '
    'INCORPORATED LLC PLC AG GMBH SARL SAC LTDA LIMITADA ANONIMA SOCIEDAD KK K SLP BHD SDN OY '
    'AB AS KG BVBA SNC SCA SCS EURL SASU PTY'.split()
)
STOP_TOKENS = set('DE DEL LA EL LOS LAS Y AND EN OF THE TO A POR PARA C'.split())
GENERIC_TOKENS = set(
    'GROUP GRUPO HOLDING HOLDINGS INTERNATIONAL INTERNACIONAL GLOBAL EUROPE EUROPA INDIA SPAIN ESPANA '
    'JAPAN JAPON NETHERLANDS NEDERLAND BELGIUM BELGICA ITALY ITALIA PRODUCTS PRODUCTOS SERVICES SERVICIOS '
    'SOLUTIONS SOLUCIONES INDUSTRIAL INDUSTRIES INDUSTRIA TECHNOLOGIES TECHNOLOGY TECH MANUFACTURING MARKETING '
    'TRADING EXPORT IMPORT LOGISTICS LOGISTICA PHARMA CHEMICAL CHEMICALS COSMETICS COSMETICA LABORATORIOS '
    'LABORATORIO DISTRIBUTION DISTRIBUCION FREIGHT'.split()
)

REPLACEMENTS = [
    ('&', ' AND '),
    ('DISE O', 'DISENO'),
    ('DISEO', 'DISENO'),
    ('DISTRIBUCIN', 'DISTRIBUCION'),
    ('S.L.', ' SL '),
    ('S. L.', ' SL '),
    ('S.A.U.', ' SAU '),
    ('S.A.', ' SA '),
    ('B.V.', ' BV '),
    ('S.P.A.', ' SPA '),
    ('S.R.L.', ' SRL '),
    ('CO.,LTD.', ' CO LTD '),
    ('CO., LTD.', ' CO LTD '),
    ('CO.,LTD', ' CO LTD '),
    ('PVT.', ' PVT '),
    ('K.K.', ' KK '),
    ('Y/O', ' '),
]

NOISE_PHRASES = [
    'POR DESIGNACION', 'PARQUE EM', 'CARRETERA', 'MARITIMO', 'INFINI', 'RESOL GRUPA',
    'C COLON', 'C MERCADERS', '7 1 NIH', 'NIH', 'B66857160',
]

LEGAL_PHRASES = [
    'SOCIEDAD LIMITADA', 'SOCIEDAD ANONIMA', 'PRIVATE LIMITED', 'PVT LTD', 'COMPANY LIMITED',
    'CO LTD', 'PUBLIC LIMITED COMPANY', 'LIMITED LIABILITY COMPANY',
]


def sin_acentos_upper(x: Any) -> str:
    if pd.isna(x):
        return ''
    s = str(x).upper().strip()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return s


def normalizar_razon_social(x: Any, *, remove_legal: bool = True) -> str:
    s = sin_acentos_upper(x)
    if not s:
        return ''
    for a, b in REPLACEMENTS:
        s = s.replace(a, b)
    s = re.sub(r'\([^)]*\)', ' ', s)
    s = re.sub(r'\b[A-Z]\d{7,}\b', ' ', s)
    s = re.sub(r'\b\d{2,}[-,/].*$', ' ', s)
    s = re.sub(r'[^A-Z0-9]+', ' ', s)
    s = f' {s} '
    for p in NOISE_PHRASES:
        s = s.replace(f' {p} ', ' ')
    for p in LEGAL_PHRASES:
        s = s.replace(f' {p} ', ' ')
    toks = []
    for t in s.split():
        if remove_legal and t in LEGAL_TOKENS:
            continue
        if t in STOP_TOKENS:
            continue
        toks.append(t)
    return re.sub(r'\s+', ' ', ' '.join(toks)).strip()


def compactar_key(s: Any) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(s or ''))


def tokens_key(s: str) -> list[str]:
    return [t for t in str(s).split() if len(t) >= 2]


def stem_token(t: str) -> str:
    if len(t) > 4 and t.endswith('ES'):
        return t[:-2]
    if len(t) > 3 and t.endswith('S'):
        return t[:-1]
    return t


def token_set(s: str) -> set[str]:
    return {stem_token(t) for t in tokens_key(s)}


def distinctive_token_set(s: str) -> set[str]:
    return {stem_token(t) for t in tokens_key(s) if len(t) >= 3 and t not in GENERIC_TOKENS and t not in LEGAL_TOKENS}


def subsector_por_partida(partida: Any) -> str:
    s = '' if pd.isna(partida) else re.sub(r'\D', '', str(partida))
    for pref, sub in sorted(SUBSECTOR_POR_PARTIDA_PREFIX.items(), key=lambda kv: len(kv[0]), reverse=True):
        if s.startswith(pref):
            return sub
    return 'No clasificado'


def pais_es_a_en(pais: Any) -> str:
    if pd.isna(pais):
        return ''
    raw = str(pais).strip()
    return PAIS_ES_A_EN.get(raw, PAIS_ES_A_EN.get(sin_acentos_upper(raw).title(), raw))


def pais_compatible(pais_b1_en: Any, pais_b2_en: Any) -> bool | None:
    a = '' if pd.isna(pais_b1_en) else str(pais_b1_en).strip()
    b = '' if pd.isna(pais_b2_en) else str(pais_b2_en).strip()
    if not a or not b:
        return None
    return sin_acentos_upper(a) == sin_acentos_upper(b)


def cargar_base1(config: ConfigNearshoring, rutas: RutasNearshoring) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not rutas.archivo_base1.exists():
        raise FileNotFoundError(f'No existe Base 1: {rutas.archivo_base1}')
    xl = pd.ExcelFile(rutas.archivo_base1)
    frames: list[pd.DataFrame] = []
    criterios = pd.DataFrame()
    for sheet in xl.sheet_names:
        df = pd.read_excel(rutas.archivo_base1, sheet_name=sheet, dtype=str)
        df.columns = [str(c).strip() for c in df.columns]
        if sheet in config.hojas_excluir_base1:
            criterios = df.copy()
            continue
        if config.col_base1_empresa not in df.columns:
            continue
        tmp = df.copy()
        tmp['_HOJA_BASE1'] = sheet
        tmp['FUENTE'] = 'BASE1_EXPORTADORES'
        tmp['ID_BASE1'] = [f'B1_{i:05d}' for i in range(len(frames), len(frames) + len(tmp))]
        tmp[config.col_nombre_raw] = tmp[config.col_base1_empresa]
        if 'País de Empresa' in tmp.columns:
            tmp['PAIS_ORIGEN_ES'] = tmp['País de Empresa'].fillna(sheet)
        else:
            tmp['PAIS_ORIGEN_ES'] = sheet
        tmp['PAIS_ORIGEN_EN'] = tmp['PAIS_ORIGEN_ES'].map(pais_es_a_en)
        tmp['SUBSECTOR_ESTIMADO'] = tmp.get('Partida', pd.Series(index=tmp.index, dtype='object')).map(subsector_por_partida)
        frames.append(tmp)
    if not frames:
        raise ValueError('No se encontraron hojas válidas en Base 1 con la columna Empresa.')
    b1 = pd.concat(frames, ignore_index=True)
    b1['ID_BASE1'] = [f'B1_{i:05d}' for i in range(len(b1))]
    b1 = b1[b1[config.col_nombre_raw].notna() & b1[config.col_nombre_raw].astype(str).str.strip().ne('')].copy()
    return b1, criterios


def cargar_base2(config: ConfigNearshoring, rutas: RutasNearshoring) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not rutas.archivo_base2.exists():
        raise FileNotFoundError(f'No existe Base 2: {rutas.archivo_base2}')

    fdi = pd.read_excel(rutas.archivo_base2, sheet_name=config.hoja_fdi, header=config.header_fdi, dtype=str)
    fdi.columns = [str(c).strip() for c in fdi.columns]
    if config.col_fdi_empresa not in fdi.columns:
        raise KeyError(f'No encontré {config.col_fdi_empresa!r} en hoja {config.hoja_fdi}. Columnas: {list(fdi.columns)}')
    fdi = fdi[fdi[config.col_fdi_empresa].notna() & fdi[config.col_fdi_empresa].astype(str).str.strip().ne('')].copy()
    fdi['FUENTE_BASE2'] = 'FDI_MARKETS'
    fdi['ID_BASE2'] = [f'FDI_{i:05d}' for i in range(len(fdi))]
    fdi[config.col_nombre_raw] = fdi[config.col_fdi_empresa]
    fdi['PAIS_ORIGEN_EN'] = fdi.get('HQ country', '')
    fdi['CONTACTO_WEBSITE'] = fdi.get('Website', '')
    fdi['CONTACTO_EMAIL'] = fdi.get('Company email', '')
    fdi['CONTACTO_TELEFONO'] = fdi.get('Company telephone #', '')

    orbis = pd.read_excel(rutas.archivo_base2, sheet_name=config.hoja_orbis, dtype=str)
    orbis.columns = [str(c).strip() for c in orbis.columns]
    if config.col_orbis_empresa not in orbis.columns:
        raise KeyError(f'No encontré {config.col_orbis_empresa!r} en hoja {config.hoja_orbis}. Columnas: {list(orbis.columns)}')
    orbis = orbis[orbis[config.col_orbis_empresa].notna() & orbis[config.col_orbis_empresa].astype(str).str.strip().ne('')].copy()
    orbis['FUENTE_BASE2'] = 'ORBIS_CROSSBORDER'
    orbis['ID_BASE2'] = [f'ORB_{i:05d}' for i in range(len(orbis))]
    orbis[config.col_nombre_raw] = orbis[config.col_orbis_empresa]
    orbis['PAIS_ORIGEN_EN'] = orbis.get('Country', '')
    orbis['CONTACTO_WEBSITE'] = orbis.get('Website address', '')
    orbis['CONTACTO_EMAIL'] = orbis.get('E-mail address', '')
    orbis['CONTACTO_TELEFONO'] = orbis.get('Telephone number', '')

    keep_common = ['FUENTE_BASE2', 'ID_BASE2', config.col_nombre_raw, 'PAIS_ORIGEN_EN', 'CONTACTO_WEBSITE', 'CONTACTO_EMAIL', 'CONTACTO_TELEFONO']
    b2 = pd.concat([fdi, orbis], ignore_index=True, sort=False)
    return b2, fdi, orbis


def preparar_campos_match(df: pd.DataFrame, config: ConfigNearshoring) -> pd.DataFrame:
    out = df.copy()
    out[config.col_nit_tecnico] = ''
    out[config.col_nombre_match] = out[config.col_nombre_raw].map(lambda x: normalizar_razon_social(x, remove_legal=True))
    out['RAZON_SOCIAL_DISPLAY_CLEAN'] = out[config.col_nombre_raw].map(lambda x: normalizar_razon_social(x, remove_legal=False))
    out['KEY_COMPACTA'] = out[config.col_nombre_match].map(compactar_key)
    out['TOKENS_MATCH'] = out[config.col_nombre_match].map(lambda s: ' '.join(sorted(token_set(s))))
    out['TOKENS_DISTINTIVOS'] = out[config.col_nombre_match].map(lambda s: ' '.join(sorted(distinctive_token_set(s))))
    out['N_TOKENS'] = out[config.col_nombre_match].map(lambda s: len(token_set(s)))
    out['N_TOKENS_DISTINTIVOS'] = out[config.col_nombre_match].map(lambda s: len(distinctive_token_set(s)))
    return out


def inicializar_rues_similarity(config: ConfigNearshoring):
    try:
        instalar_rues_linker(config.ruta_paquete)
        from record_linkage.engine.similarity import SimilarityCalculator
        return SimilarityCalculator()
    except Exception as exc:
        print(f'⚠️ No se pudo inicializar SimilarityCalculator de rues-linker: {type(exc).__name__}: {exc}')
        print('   Se usará fallback rapidfuzz; revise la ruta_paquete si esto ocurre en Colab.')
        return None


def score_pair(nombre1: str, nombre2: str, rues_sim=None, config: ConfigNearshoring | None = None) -> dict[str, Any]:
    config = config or ConfigNearshoring()
    k1 = str(nombre1 or '').strip()
    k2 = str(nombre2 or '').strip()
    c1 = compactar_key(k1)
    c2 = compactar_key(k2)
    if not k1 or not k2 or not c1 or not c2:
        return {'decision_algoritmica': 'NO_MATCH', 'criterio': 'EMPTY_KEY', 'score_final': 0.0}

    set1 = token_set(k1)
    set2 = token_set(k2)
    ds1 = distinctive_token_set(k1)
    ds2 = distinctive_token_set(k2)
    inter = set1 & set2
    dinter = ds1 & ds2
    union = set1 | set2

    ratio = float(fuzz.ratio(k1, k2))
    token_sort = float(fuzz.token_sort_ratio(k1, k2))
    token_set_ratio = float(fuzz.token_set_ratio(k1, k2))
    partial = float(fuzz.partial_ratio(k1, k2))
    compact_ratio = float(fuzz.ratio(c1, c2))
    rues_score = float(rues_sim.name_similarity(k1, k2)) if rues_sim is not None else max(token_set_ratio, ratio) / 100.0

    jaccard = len(inter) / max(1, len(union))
    containment = max(len(inter) / max(1, len(set1)), len(inter) / max(1, len(set2)))
    distinctive_containment = 0.0
    if ds1 or ds2:
        distinctive_containment = max(len(dinter) / max(1, len(ds1)), len(dinter) / max(1, len(ds2)))

    decision = 'NO_MATCH'
    criterio = 'SIN_REGLA'
    score_final = max(ratio, token_sort, compact_ratio, rues_score * 100.0)

    # Regla 1: llave compacta igual tras normalización societaria.
    if c1 == c2 and len(c1) >= config.min_largo_compacta_match:
        if len(set1) == 1 and len(set2) == 1 and not config.permitir_match_seguro_un_token_exacto:
            decision = 'MATCH_PROBABLE_REVISION'
            criterio = 'EXACT_ONE_TOKEN_REVIEW'
            score_final = 96.0
        else:
            decision = 'MATCH_SEGURO'
            criterio = 'EXACT_COMPACT_KEY'
            score_final = 100.0

    # Regla 2: mismo conjunto de tokens con plural/singular o reordenamiento.
    elif set1 == set2 and len(set1) >= 1 and max(ratio, token_sort, compact_ratio) >= 88.0:
        decision = 'MATCH_SEGURO'
        criterio = 'SAME_TOKEN_SET_HIGH_RATIO'
        score_final = max(score_final, 96.0)

    # Regla 3: contención fuerte con al menos dos tokens distintivos.
    elif (
        len(dinter) >= 2
        and distinctive_containment >= 0.95
        and max(len(ds1), len(ds2)) <= len(dinter) + 1
        and token_set_ratio >= 92.0
        and min(len(c1), len(c2)) >= 8
    ):
        decision = 'MATCH_SEGURO'
        criterio = 'DISTINCTIVE_TOKEN_CONTAINMENT'
        score_final = max(score_final, 94.0)

    # Regla 4: similitud alta por ratio/secuencia + token distintivo común.
    elif (
        (ratio >= config.umbral_ratio_seguro or compact_ratio >= config.umbral_ratio_seguro or rues_score >= config.umbral_rues_seguro)
        and len(dinter) >= 1
        and min(len(c1), len(c2)) >= 6
    ):
        decision = 'MATCH_SEGURO'
        criterio = 'HIGH_STRING_RATIO'
        score_final = max(score_final, 94.0)

    # Reglas de revisión: alta similitud no suficiente para automático.
    elif (
        len(dinter) >= 1
        and (ratio >= config.umbral_ratio_revision or token_sort >= 90.0 or rues_score >= config.umbral_rues_revision)
        and min(len(c1), len(c2)) >= 8
    ):
        decision = 'MATCH_PROBABLE_REVISION'
        criterio = 'HIGH_SIMILARITY_REVIEW'
        score_final = max(score_final, 88.0)

    elif (
        config.permitir_revision_un_token
        and len(inter) >= 1
        and min(len(set1), len(set2)) == 1
        and containment >= 1.0
        and min(len(c1), len(c2)) >= 4
    ):
        decision = 'MATCH_PROBABLE_REVISION'
        criterio = 'ONE_TOKEN_CONTAINMENT_REVIEW'
        score_final = max(score_final, 85.0)

    return {
        'decision_algoritmica': decision,
        'criterio_match': criterio,
        'score_final': round(float(score_final), 4),
        'score_rues_linker': round(float(rues_score), 6),
        'ratio': round(ratio, 4),
        'token_sort_ratio': round(token_sort, 4),
        'token_set_ratio': round(token_set_ratio, 4),
        'partial_ratio': round(partial, 4),
        'compact_ratio': round(compact_ratio, 4),
        'jaccard_tokens': round(float(jaccard), 6),
        'containment_tokens': round(float(containment), 6),
        'containment_distintivos': round(float(distinctive_containment), 6),
        'tokens_overlap': ' '.join(sorted(inter)),
        'tokens_distintivos_overlap': ' '.join(sorted(dinter)),
        'key1': k1,
        'key2': k2,
    }


def clasificar_decision_final(row: pd.Series, config: ConfigNearshoring) -> str:
    base = row.get('decision_algoritmica', 'NO_MATCH')
    pais = row.get('PAIS_COMPATIBLE')
    if base == 'MATCH_SEGURO':
        if pais is False and config.exigir_pais_compatible_para_seguro_juridico:
            return 'MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR'
        return 'MATCH_SEGURO_MISMA_EMPRESA'
    if base == 'MATCH_PROBABLE_REVISION':
        return 'MATCH_PROBABLE_REVISION_MANUAL'
    return 'NO_MATCH'


def generar_pares_candidatos(b1: pd.DataFrame, b2: pd.DataFrame, config: ConfigNearshoring, rues_sim=None) -> pd.DataFrame:
    b1_cols = ['ID_BASE1', 'FUENTE', 'PAIS_ORIGEN_ES', 'PAIS_ORIGEN_EN', 'SUBSECTOR_ESTIMADO', config.col_base1_empresa, config.col_nombre_raw, config.col_nombre_match, 'KEY_COMPACTA']
    b1_cols = [c for c in b1_cols if c in b1.columns]
    b2_cols = ['ID_BASE2', 'FUENTE_BASE2', 'PAIS_ORIGEN_EN', config.col_nombre_raw, config.col_nombre_match, 'KEY_COMPACTA', 'CONTACTO_WEBSITE', 'CONTACTO_EMAIL', 'CONTACTO_TELEFONO']
    b2_cols = [c for c in b2_cols if c in b2.columns]
    rows: list[dict[str, Any]] = []
    recs1 = b1[b1[config.col_nombre_match].astype(str).str.strip().ne('')][b1_cols].to_dict('records')
    recs2 = b2[b2[config.col_nombre_match].astype(str).str.strip().ne('')][b2_cols].to_dict('records')
    for r1 in recs1:
        for r2 in recs2:
            sc = score_pair(r1[config.col_nombre_match], r2[config.col_nombre_match], rues_sim=rues_sim, config=config)
            if sc['decision_algoritmica'] == 'NO_MATCH':
                continue
            pc = pais_compatible(r1.get('PAIS_ORIGEN_EN'), r2.get('PAIS_ORIGEN_EN'))
            row = {
                'ID_BASE1': r1.get('ID_BASE1'),
                'EMPRESA_BASE1': r1.get(config.col_base1_empresa),
                'PAIS_BASE1_ES': r1.get('PAIS_ORIGEN_ES'),
                'PAIS_BASE1_EN': r1.get('PAIS_ORIGEN_EN'),
                'SUBSECTOR_BASE1': r1.get('SUBSECTOR_ESTIMADO'),
                'NOMBRE_BASE1_RAW': r1.get(config.col_nombre_raw),
                'NOMBRE_BASE1_MATCH': r1.get(config.col_nombre_match),
                'KEY_BASE1': r1.get('KEY_COMPACTA'),
                'ID_BASE2': r2.get('ID_BASE2'),
                'FUENTE_BASE2': r2.get('FUENTE_BASE2'),
                'PAIS_BASE2_EN': r2.get('PAIS_ORIGEN_EN'),
                'NOMBRE_BASE2_RAW': r2.get(config.col_nombre_raw),
                'NOMBRE_BASE2_MATCH': r2.get(config.col_nombre_match),
                'KEY_BASE2': r2.get('KEY_COMPACTA'),
                'CONTACTO_WEBSITE': r2.get('CONTACTO_WEBSITE'),
                'CONTACTO_EMAIL': r2.get('CONTACTO_EMAIL'),
                'CONTACTO_TELEFONO': r2.get('CONTACTO_TELEFONO'),
                'PAIS_COMPATIBLE': pc,
            }
            row.update(sc)
            rows.append(row)
    if not rows:
        return pd.DataFrame()
    pares = pd.DataFrame(rows)
    pares['DECISION_FINAL'] = pares.apply(lambda r: clasificar_decision_final(r, config), axis=1)
    prioridad = {
        'MATCH_SEGURO_MISMA_EMPRESA': 1,
        'MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR': 2,
        'MATCH_PROBABLE_REVISION_MANUAL': 3,
        'NO_MATCH': 9,
    }
    pares['_PRIORIDAD'] = pares['DECISION_FINAL'].map(prioridad).fillna(9).astype(int)
    pares['_CONTACTOS_N'] = pares[['CONTACTO_WEBSITE', 'CONTACTO_EMAIL', 'CONTACTO_TELEFONO']].notna().sum(axis=1)
    pares = pares.sort_values(['_PRIORIDAD', 'score_final', '_CONTACTOS_N', 'ID_BASE1'], ascending=[True, False, False, True])
    return pares


def construir_resultado_base1(b1: pd.DataFrame, pares: pd.DataFrame, config: ConfigNearshoring) -> pd.DataFrame:
    b1_out = b1.copy()
    if pares.empty:
        b1_out['TIENE_MATCH_INVERSIONISTA'] = False
        b1_out['DECISION_FINAL'] = 'NO_MATCH'
        return b1_out
    best = pares.sort_values(['_PRIORIDAD', 'score_final', '_CONTACTOS_N'], ascending=[True, False, False]).groupby('ID_BASE1', as_index=False).head(1)
    cols_best = [
        'ID_BASE1', 'ID_BASE2', 'FUENTE_BASE2', 'NOMBRE_BASE2_RAW', 'PAIS_BASE2_EN',
        'DECISION_FINAL', 'criterio_match', 'score_final', 'score_rues_linker',
        'CONTACTO_WEBSITE', 'CONTACTO_EMAIL', 'CONTACTO_TELEFONO', 'PAIS_COMPATIBLE',
    ]
    cols_best = [c for c in cols_best if c in best.columns]
    out = b1_out.merge(best[cols_best], on='ID_BASE1', how='left')
    out['TIENE_MATCH_INVERSIONISTA'] = out['DECISION_FINAL'].notna()
    out['DECISION_FINAL'] = out['DECISION_FINAL'].fillna('NO_MATCH')
    return out


def construir_controles_calidad(b1: pd.DataFrame, b2: pd.DataFrame, pares: pd.DataFrame, resultado_base1: pd.DataFrame) -> pd.DataFrame:
    controles = []
    def add(control, valor, estado='OK', detalle=''):
        controles.append({'control': control, 'valor': valor, 'estado': estado, 'detalle': detalle})
    add('filas_base1', len(b1))
    add('empresas_base1_unicas_raw', b1['RAZON_SOCIAL_RAW'].nunique(dropna=True))
    add('empresas_base1_unicas_match', b1['RAZON_SOCIAL_MATCH'].nunique(dropna=True))
    add('filas_base2', len(b2))
    add('empresas_base2_unicas_raw', b2['RAZON_SOCIAL_RAW'].nunique(dropna=True))
    add('empresas_base2_unicas_match', b2['RAZON_SOCIAL_MATCH'].nunique(dropna=True))
    add('pares_candidatos', len(pares))
    if not pares.empty:
        vc = pares['DECISION_FINAL'].value_counts(dropna=False).to_dict()
        for k, v in vc.items():
            add(f'pares_{k}', int(v))
        multiples = pares[pares['DECISION_FINAL'].isin(['MATCH_SEGURO_MISMA_EMPRESA', 'MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR'])].groupby('ID_BASE1')['ID_BASE2'].nunique()
        n_multi = int((multiples > 1).sum())
        add('base1_con_multiples_matches_alta_confianza', n_multi, 'REVISAR' if n_multi else 'OK')
        pais_mismatch = int((pares['PAIS_COMPATIBLE'] == False).sum())
        add('pares_con_pais_distinto', pais_mismatch, 'REVISAR' if pais_mismatch else 'OK')
    add('base1_con_alguna_coincidencia', int(resultado_base1['TIENE_MATCH_INVERSIONISTA'].sum()))
    return pd.DataFrame(controles)


def _colapsar_unicos(s: pd.Series, max_items: int = 10) -> str:
    vals = s.dropna().astype(str).map(str.strip).replace('', np.nan).dropna().drop_duplicates().tolist()
    vals = sorted(vals)
    if len(vals) > max_items:
        return ' | '.join(vals[:max_items]) + f' | ... (+{len(vals) - max_items})'
    return ' | '.join(vals)


def construir_correlativa(pares: pd.DataFrame) -> pd.DataFrame:
    if pares.empty:
        return pd.DataFrame()
    cols = [
        'ID_BASE1', 'EMPRESA_BASE1', 'PAIS_BASE1_ES', 'PAIS_BASE1_EN', 'SUBSECTOR_BASE1',
        'ID_BASE2', 'FUENTE_BASE2', 'NOMBRE_BASE2_RAW', 'PAIS_BASE2_EN',
        'DECISION_FINAL', 'criterio_match', 'score_final', 'score_rues_linker',
        'PAIS_COMPATIBLE', 'CONTACTO_WEBSITE', 'CONTACTO_EMAIL', 'CONTACTO_TELEFONO',
        'NOMBRE_BASE1_MATCH', 'NOMBRE_BASE2_MATCH', 'tokens_overlap', 'tokens_distintivos_overlap',
    ]
    cols = [c for c in cols if c in pares.columns]
    return pares[cols].copy()


def resumen_ejecutivo_df(config: ConfigNearshoring, b1: pd.DataFrame, b2: pd.DataFrame, pares: pd.DataFrame, resultado_base1: pd.DataFrame) -> pd.DataFrame:
    rows = [
        ('fecha_procesamiento', config.fecha_procesamiento),
        ('etiqueta_corrida', config.etiqueta_corrida),
        ('base1_filas_exportadores', len(b1)),
        ('base1_empresas_unicas_raw', b1['RAZON_SOCIAL_RAW'].nunique(dropna=True)),
        ('base2_filas_inversionistas', len(b2)),
        ('base2_empresas_unicas_raw', b2['RAZON_SOCIAL_RAW'].nunique(dropna=True)),
        ('pares_candidatos_total', len(pares)),
    ]
    if not pares.empty:
        for k, v in pares['DECISION_FINAL'].value_counts().to_dict().items():
            rows.append((f'pares_{k}', int(v)))
        rows.append(('base1_con_match_seguro_misma_empresa', int((resultado_base1['DECISION_FINAL'] == 'MATCH_SEGURO_MISMA_EMPRESA').sum())))
        rows.append(('base1_con_match_alta_confianza_pais_distinto', int((resultado_base1['DECISION_FINAL'] == 'MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR').sum())))
        rows.append(('base1_con_match_probable_revision', int((resultado_base1['DECISION_FINAL'] == 'MATCH_PROBABLE_REVISION_MANUAL').sum())))
    rows.append(('advertencia_calidad', 'Sin ground truth etiquetado no se puede garantizar F1; se prioriza precisión y auditabilidad sobre recall.'))
    return pd.DataFrame(rows, columns=['metrica', 'valor'])


def exportar_resultados(
    config: ConfigNearshoring,
    rutas: RutasNearshoring,
    b1: pd.DataFrame,
    b2: pd.DataFrame,
    pares: pd.DataFrame,
    correlativa: pd.DataFrame,
    resultado_base1: pd.DataFrame,
    controles: pd.DataFrame,
    resumen: pd.DataFrame,
    criterios: pd.DataFrame,
) -> None:
    rutas.output_dir.mkdir(parents=True, exist_ok=True)
    if config.exportar_csv:
        b1.to_csv(rutas.output_dir / 'base1_unificada.csv', index=False, encoding='utf-8-sig')
        b2.to_csv(rutas.output_dir / 'base2_unificada.csv', index=False, encoding='utf-8-sig')
        pares.to_csv(rutas.output_dir / 'pares_candidatos_auditados.csv', index=False, encoding='utf-8-sig')
        correlativa.to_csv(rutas.output_dir / 'correlativa_exportador_inversionista.csv', index=False, encoding='utf-8-sig')
        resultado_base1.to_csv(rutas.output_dir / 'base1_con_match_inversionista.csv', index=False, encoding='utf-8-sig')
        controles.to_csv(rutas.output_dir / 'control_calidad.csv', index=False, encoding='utf-8-sig')
    if config.exportar_parquet:
        b1.to_parquet(rutas.output_dir / 'base1_unificada.parquet', index=False)
        b2.to_parquet(rutas.output_dir / 'base2_unificada.parquet', index=False)
        pares.to_parquet(rutas.output_dir / 'pares_candidatos_auditados.parquet', index=False)
        correlativa.to_parquet(rutas.output_dir / 'correlativa_exportador_inversionista.parquet', index=False)
        resultado_base1.to_parquet(rutas.output_dir / 'base1_con_match_inversionista.parquet', index=False)
    if config.exportar_excel:
        with pd.ExcelWriter(rutas.excel_resultados, engine='xlsxwriter') as writer:
            resumen.to_excel(writer, sheet_name='00_resumen', index=False)
            b1.to_excel(writer, sheet_name='01_base1_unificada', index=False)
            b2.to_excel(writer, sheet_name='02_base2_unificada', index=False)
            pares.to_excel(writer, sheet_name='03_pares_candidatos', index=False)
            correlativa[correlativa['DECISION_FINAL'].eq('MATCH_SEGURO_MISMA_EMPRESA')].to_excel(writer, sheet_name='04_correlativa_segura', index=False)
            correlativa[~correlativa['DECISION_FINAL'].eq('MATCH_SEGURO_MISMA_EMPRESA')].to_excel(writer, sheet_name='05_revision_manual', index=False)
            resultado_base1.to_excel(writer, sheet_name='06_base1_con_match', index=False)
            controles.to_excel(writer, sheet_name='07_control_calidad', index=False)
            if criterios is not None and not criterios.empty:
                criterios.to_excel(writer, sheet_name='08_criterios_cruce', index=False)
            wb = writer.book
            header_fmt = wb.add_format({'bold': True, 'bg_color': '#D9EAF7', 'border': 1})
            warn_fmt = wb.add_format({'bg_color': '#FFF2CC'})
            ok_fmt = wb.add_format({'bg_color': '#E2F0D9'})
            for ws_name, ws in writer.sheets.items():
                ws.freeze_panes(1, 0)
                ws.autofilter(0, 0, 0, max(0, ws.dim_colmax or 0))
                # Header format and reasonable widths.
                try:
                    df_written = {
                        '00_resumen': resumen, '01_base1_unificada': b1, '02_base2_unificada': b2,
                        '03_pares_candidatos': pares, '04_correlativa_segura': correlativa[correlativa['DECISION_FINAL'].eq('MATCH_SEGURO_MISMA_EMPRESA')],
                        '05_revision_manual': correlativa[~correlativa['DECISION_FINAL'].eq('MATCH_SEGURO_MISMA_EMPRESA')],
                        '06_base1_con_match': resultado_base1, '07_control_calidad': controles,
                        '08_criterios_cruce': criterios,
                    }.get(ws_name)
                    if df_written is not None:
                        for col_idx, col_name in enumerate(df_written.columns):
                            ws.write(0, col_idx, col_name, header_fmt)
                            sample = df_written[col_name].astype(str).head(200).tolist() if len(df_written) else []
                            width = min(max([len(str(col_name))] + [len(x) for x in sample]) + 2, 45)
                            ws.set_column(col_idx, col_idx, width)
                        if 'DECISION_FINAL' in df_written.columns and len(df_written):
                            c = df_written.columns.get_loc('DECISION_FINAL')
                            ws.conditional_format(1, c, len(df_written), c, {'type': 'text', 'criteria': 'containing', 'value': 'SEGURA', 'format': ok_fmt})
                            ws.conditional_format(1, c, len(df_written), c, {'type': 'text', 'criteria': 'containing', 'value': 'REVISAR', 'format': warn_fmt})
                except Exception:
                    pass


def ejecutar_pipeline(config: ConfigNearshoring | None = None) -> dict[str, Any]:
    config = config or ConfigNearshoring()
    rutas = construir_rutas(config)
    rutas.output_dir.mkdir(parents=True, exist_ok=True)
    log = setup_logger(rutas.log_file)
    log.info('Iniciando Nearshoring tres fuentes')
    log.info(f'Base 1: {rutas.archivo_base1}')
    log.info(f'Base 2: {rutas.archivo_base2}')

    rues_sim = inicializar_rues_similarity(config) if config.usar_rues_similarity else None

    b1, criterios = cargar_base1(config, rutas)
    b2, fdi, orbis = cargar_base2(config, rutas)
    b1 = preparar_campos_match(b1, config)
    b2 = preparar_campos_match(b2, config)

    log.info(f'Base 1 preparada: {len(b1):,} filas · {b1[config.col_nombre_raw].nunique():,} empresas raw únicas')
    log.info(f'Base 2 preparada: {len(b2):,} filas · {b2[config.col_nombre_raw].nunique():,} empresas raw únicas')

    pares = generar_pares_candidatos(b1, b2, config, rues_sim=rues_sim)
    correlativa = construir_correlativa(pares)
    resultado_base1 = construir_resultado_base1(b1, pares, config)
    controles = construir_controles_calidad(b1, b2, pares, resultado_base1)
    resumen = resumen_ejecutivo_df(config, b1, b2, pares, resultado_base1)

    exportar_resultados(config, rutas, b1, b2, pares, correlativa, resultado_base1, controles, resumen, criterios)

    log.info(f'Pares candidatos: {len(pares):,}')
    if not pares.empty:
        log.info('Decisiones: ' + json.dumps(pares['DECISION_FINAL'].value_counts().to_dict(), ensure_ascii=False))
    log.info(f'Salidas: {rutas.output_dir}')
    return {
        'config': config,
        'rutas': rutas,
        'base1': b1,
        'base2': b2,
        'pares': pares,
        'correlativa': correlativa,
        'resultado_base1': resultado_base1,
        'controles': controles,
        'resumen': resumen,
        'criterios': criterios,
    }



## B. Configuración editable

La ruta solicitada para la librería se conserva: `/content/drive/MyDrive/ProColombia/record_linkage`.

La carpeta de datos usa la fecha indicada para la corrida: `/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29`.


In [3]:

# ════════════════════════════════════════════════════════════════════════
# B.1 — Configuración de corrida
# ════════════════════════════════════════════════════════════════════════

montar_drive_con_retry()

CONFIG = ConfigNearshoring(
    fecha_procesamiento='2026-06-29',
    etiqueta_corrida='NEARSHORING_TRES_FUENTES',
    ruta_paquete=Path('/content/drive/MyDrive/ProColombia/record_linkage') if EN_COLAB else Path('/mnt/data/rues_unpacked/rues-linker-main'),
    dir_datos=Path('/content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29') if EN_COLAB else Path('/mnt/data'),
    archivo_base1='Base 1_Empresas exportadoras a USA en 2025 por países y productos priorizados.xlsx',
    archivo_base2='Base 2_Descargas bases de inversión.xlsx',
    hoja_fdi='fDiMarkets',
    hoja_orbis='Orbis Crossboarder',
    header_fdi=2,

    # Decisión de calidad: el país no bloquea el candidato, pero sí degrada
    # el match exacto a revisión si país Base 1 y país Base 2 son distintos.
    exigir_pais_compatible_para_seguro_juridico=True,

    # Modo conservador: no convertir coincidencias de un solo token en match seguro.
    permitir_revision_un_token=True,
    permitir_match_seguro_un_token_exacto=True,

    exportar_excel=True,
    exportar_csv=True,
    exportar_parquet=False,
)

RUTAS = construir_rutas(CONFIG)
print('📁 Base 1:', RUTAS.archivo_base1)
print('📁 Base 2:', RUTAS.archivo_base2)
print('📤 Salida:', RUTAS.output_dir)


Mounted at /content/drive
✅ Drive montado (intento 1/3).
📁 Base 1: /content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29/Base 1_Empresas exportadoras a USA en 2025 por países y productos priorizados.xlsx
📁 Base 2: /content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29/Base 2_Descargas bases de inversión.xlsx
📤 Salida: /content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29/salidas_nearshoring_tres_fuentes_20260630_043708



## C. Ejecución completa

Esta celda lee ambas bases, normaliza nombres, activa `rues-linker`, genera pares candidatos, clasifica decisiones y exporta resultados.


In [4]:

# ════════════════════════════════════════════════════════════════════════
# C.1 — Ejecutar pipeline
# ════════════════════════════════════════════════════════════════════════

RESULTADO = ejecutar_pipeline(CONFIG)

print('\n✅ Corrida terminada')
print('📤 Carpeta de salida:', RESULTADO['rutas'].output_dir)
print('📘 Excel consolidado:', RESULTADO['rutas'].excel_resultados)

display(RESULTADO['resumen'])


2026-06-30 04:37:12 [INFO   ] Iniciando Nearshoring tres fuentes
2026-06-30 04:37:12 [INFO   ] Base 1: /content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29/Base 1_Empresas exportadoras a USA en 2025 por países y productos priorizados.xlsx
2026-06-30 04:37:12 [INFO   ] Base 2: /content/drive/MyDrive/ProColombia/Deduplicación/datos/2026-06-29/Base 2_Descargas bases de inversión.xlsx
▶ Instalando repo editable: /usr/bin/python3 -m pip install -q -e /content/drive/MyDrive/ProColombia/record_linkage[excel]
✅ record_linkage importado desde: /content/drive/MyDrive/ProColombia/record_linkage/src/record_linkage/__init__.py
2026-06-30 04:38:33 [INFO   ] Base 1 preparada: 107 filas · 103 empresas raw únicas
2026-06-30 04:38:33 [INFO   ] Base 2 preparada: 650 filas · 650 empresas raw únicas
2026-06-30 04:38:35 [INFO   ] Pares candidatos: 5
2026-06-30 04:38:35 [INFO   ] Decisiones: {"MATCH_SEGURO_MISMA_EMPRESA": 3, "MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR": 1, "MATCH_PROBABLE_REVISI

,metrica,valor
0,fecha_procesamiento,2026-06-29
1,etiqueta_corrida,NEARSHORING_TRES_FUENTES
2,base1_filas_exportadores,107
3,base1_empresas_unicas_raw,103
4,base2_filas_inversionistas,650
5,base2_empresas_unicas_raw,650
6,pares_candidatos_total,5
7,pares_MATCH_SEGURO_MISMA_EMPRESA,3
8,pares_MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR,1
9,pares_MATCH_PROBABLE_REVISION_MANUAL,1



## D. Revisión de resultados principales

Use estas vistas antes de entregar la correlativa. No tome como match automático lo que el notebook marca como revisión.


In [5]:

# D.1 — Correlativa completa de candidatos
cols_vista = [
    'EMPRESA_BASE1', 'PAIS_BASE1_ES', 'SUBSECTOR_BASE1',
    'NOMBRE_BASE2_RAW', 'FUENTE_BASE2', 'PAIS_BASE2_EN',
    'DECISION_FINAL', 'criterio_match', 'score_final', 'score_rues_linker',
    'PAIS_COMPATIBLE', 'CONTACTO_WEBSITE', 'CONTACTO_EMAIL', 'CONTACTO_TELEFONO'
]
cols_vista = [c for c in cols_vista if c in RESULTADO['pares'].columns]

display(RESULTADO['pares'][cols_vista].sort_values(['DECISION_FINAL', 'score_final'], ascending=[True, False]))


,EMPRESA_BASE1,PAIS_BASE1_ES,SUBSECTOR_BASE1,NOMBRE_BASE2_RAW,FUENTE_BASE2,PAIS_BASE2_EN,DECISION_FINAL,criterio_match,score_final,score_rues_linker,PAIS_COMPATIBLE,CONTACTO_WEBSITE,CONTACTO_EMAIL,CONTACTO_TELEFONO
3,DRYLOCK TECHNOLOGIES SL,España,Envases y empaques de papel y cartón,Drylock Technologies,FDI_MARKETS,Belgium,MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR,EXACT_COMPACT_KEY,100.0,1.00,False,drylocktechnologies.com,info@drylock.eu,+3252559560
4,BOSAL INDUSTRIAL ZARAGOZA SA,España,Envases y empaques de papel y cartón,Bosal,FDI_MARKETS,Belgium,MATCH_PROBABLE_REVISION_MANUAL,ONE_TOKEN_CONTAINMENT_REVIEW,97.0,0.97,False,bosal.com/,info.bnl@eur.bosal.com,+31347362911
0,ANTONIO PUIG SA POR DESIGNACION,España,Cosméticos,ANTONIO PUIG SAU,ORBIS_CROSSBORDER,Spain,MATCH_SEGURO_MISMA_EMPRESA,EXACT_COMPACT_KEY,100.0,1.00,True,www.puig.com,NaN,+34 934007000
1,"BEAUTYGE, S.L.",España,Cosméticos,BEAUTYGE S.L.,ORBIS_CROSSBORDER,Spain,MATCH_SEGURO_MISMA_EMPRESA,EXACT_COMPACT_KEY,100.0,1.00,True,www.revlon.com,NaN,+34 934009300
2,GERMAINE DE CAPUCCINI SA,España,Cosméticos,"GERMAINE DE CAPUCCINI, SAU",ORBIS_CROSSBORDER,Spain,MATCH_SEGURO_MISMA_EMPRESA,EXACT_COMPACT_KEY,100.0,1.00,True,www.germainedecapuccini.es,NaN,+34 965547070


In [6]:

# D.2 — Solo matches jurídicamente seguros: misma empresa + país compatible
correlativa_segura = RESULTADO['correlativa'][
    RESULTADO['correlativa']['DECISION_FINAL'].eq('MATCH_SEGURO_MISMA_EMPRESA')
].copy()

display(correlativa_segura)


,ID_BASE1,EMPRESA_BASE1,PAIS_BASE1_ES,PAIS_BASE1_EN,SUBSECTOR_BASE1,ID_BASE2,FUENTE_BASE2,NOMBRE_BASE2_RAW,PAIS_BASE2_EN,DECISION_FINAL,...,score_final,score_rues_linker,PAIS_COMPATIBLE,CONTACTO_WEBSITE,CONTACTO_EMAIL,CONTACTO_TELEFONO,NOMBRE_BASE1_MATCH,NOMBRE_BASE2_MATCH,tokens_overlap,tokens_distintivos_overlap
0,B1_00030,ANTONIO PUIG SA POR DESIGNACION,España,Spain,Cosméticos,ORB_00047,ORBIS_CROSSBORDER,ANTONIO PUIG SAU,Spain,MATCH_SEGURO_MISMA_EMPRESA,...,100.0,1.0,True,www.puig.com,NaN,+34 934007000,ANTONIO PUIG,ANTONIO PUIG,ANTONIO PUIG,ANTONIO PUIG
1,B1_00053,"BEAUTYGE, S.L.",España,Spain,Cosméticos,ORB_00044,ORBIS_CROSSBORDER,BEAUTYGE S.L.,Spain,MATCH_SEGURO_MISMA_EMPRESA,...,100.0,1.0,True,www.revlon.com,NaN,+34 934009300,BEAUTYGE,BEAUTYGE,BEAUTYGE,BEAUTYGE
2,B1_00063,GERMAINE DE CAPUCCINI SA,España,Spain,Cosméticos,ORB_00063,ORBIS_CROSSBORDER,"GERMAINE DE CAPUCCINI, SAU",Spain,MATCH_SEGURO_MISMA_EMPRESA,...,100.0,1.0,True,www.germainedecapuccini.es,NaN,+34 965547070,GERMAINE CAPUCCINI,GERMAINE CAPUCCINI,CAPUCCINI GERMAINE,CAPUCCINI GERMAINE


In [7]:

# D.3 — Casos que requieren validación manual / OFICOM
revision_manual = RESULTADO['correlativa'][
    ~RESULTADO['correlativa']['DECISION_FINAL'].eq('MATCH_SEGURO_MISMA_EMPRESA')
].copy()

display(revision_manual)


,ID_BASE1,EMPRESA_BASE1,PAIS_BASE1_ES,PAIS_BASE1_EN,SUBSECTOR_BASE1,ID_BASE2,FUENTE_BASE2,NOMBRE_BASE2_RAW,PAIS_BASE2_EN,DECISION_FINAL,...,score_final,score_rues_linker,PAIS_COMPATIBLE,CONTACTO_WEBSITE,CONTACTO_EMAIL,CONTACTO_TELEFONO,NOMBRE_BASE1_MATCH,NOMBRE_BASE2_MATCH,tokens_overlap,tokens_distintivos_overlap
3,B1_00081,DRYLOCK TECHNOLOGIES SL,España,Spain,Envases y empaques de papel y cartón,FDI_00111,FDI_MARKETS,Drylock Technologies,Belgium,MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR,...,100.0,1.00,False,drylocktechnologies.com,info@drylock.eu,+3252559560,DRYLOCK TECHNOLOGIES,DRYLOCK TECHNOLOGIES,DRYLOCK TECHNOLOGI,DRYLOCK
4,B1_00090,BOSAL INDUSTRIAL ZARAGOZA SA,España,Spain,Envases y empaques de papel y cartón,FDI_00493,FDI_MARKETS,Bosal,Belgium,MATCH_PROBABLE_REVISION_MANUAL,...,97.0,0.97,False,bosal.com/,info.bnl@eur.bosal.com,+31347362911,BOSAL INDUSTRIAL ZARAGOZA,BOSAL,BOSAL,BOSAL


In [8]:

# D.4 — Controles de calidad
# Si aparece estado REVISAR, no lo ignore: normalmente indica país distinto,
# match de un solo token o múltiples candidatos para el mismo exportador.

display(RESULTADO['controles'])


,control,valor,estado,detalle
0,filas_base1,107,OK,
1,empresas_base1_unicas_raw,103,OK,
2,empresas_base1_unicas_match,88,OK,
3,filas_base2,650,OK,
4,empresas_base2_unicas_raw,650,OK,
5,empresas_base2_unicas_match,647,OK,
6,pares_candidatos,5,OK,
7,pares_MATCH_SEGURO_MISMA_EMPRESA,3,OK,
8,pares_MATCH_ALTA_CONFIANZA_PAIS_DISTINTO_REVISAR,1,OK,
9,pares_MATCH_PROBABLE_REVISION_MANUAL,1,OK,



## E. Interpretación operacional

El Excel consolidado se guarda en la carpeta de salida con estas hojas:

- `00_resumen`: métricas de corrida.
- `01_base1_unificada`: Base 1 pegada hacia abajo, una hoja por país original.
- `02_base2_unificada`: fDiMarkets + Orbis Crossboarder en una sola base.
- `03_pares_candidatos`: todos los candidatos aceptados por la política de decisión.
- `04_correlativa_segura`: matches automáticos conservadores.
- `05_revision_manual`: candidatos que no deben usarse sin validación.
- `06_base1_con_match`: Base 1 con el mejor match seleccionado por prioridad.
- `07_control_calidad`: pruebas automáticas de consistencia.
- `08_criterios_cruce`: criterios de descarga, si la hoja existe en Base 1.
